In [1]:
# Set up the OpenAI client

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
# Load the RAG helper and the FAQ data, and build the index

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
# Set up the RAG assistant

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
print(assistant.rag("How do I run Ollama locally?"))

To run Ollama locally, first install it from [https://ollama.com/download](https://ollama.com/download) for your operating system:

- **macOS**: download and install the `.pkg`
- **Windows**: download and install the `.msi`
- **Linux**: run:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

After installation, open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can use:

```bash
curl http://localhost:11434
```

If you get a connection refused error while prompting Ollama, restart the server with:

```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [9]:
print(assistant.rag("How do I run Olama locally?"))

I couldn’t find any FAQ entry about **Ollama** specifically.

The closest relevant note is that you **can run the course locally** instead of using Codespaces, as long as you’re comfortable setting up the needed tools like **Python, `uv`, Jupyter, Docker, and anything else required for the module**. If you do run locally, **document your setup and keep it reproducible**.

If you meant a specific setup step for Ollama, please share the module or workflow you’re using.


#### The problem illustrated above is that a simple typo can cause it to fail!

##### To fix, put the LLM in charge of the search, so that it fixes any typos and tries again...

In [ ]:
# Example of sending a message to the OpenAI API and getting a response
# Note that no context is provided here, so the model will respond based on its general knowledge.

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Just send me:\n- the course name,\n- the school/platform,\n- and any deadline or enrollment page you found.\n\nIn general, check for:\n- open enrollment / late registration,\n- prerequisites,\n- whether there’s a waitlist,\n- and whether the course is self-paced or has a fixed start date.\n\nIf you already have a message you want to send the instructor or admin, I can draft one for you.'

The model answers from its general knowledge, something like "it depends on the course" or "check the course website". It doesn't know about our FAQ, so the answer is vague and not helpful. This is exactly why we need RAG, and why we want to hand the model a tool.

Here's a copy of the search function from the RAGBase class, which you can use to search the index directly...

In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [12]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The description is the most important field, because the model reads it to decide when to call the function. parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

Now we send the same question as before, but this time we include the tool in the request:

In [17]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join registration"}', call_id='call_X2FvALajrHjECxIPM4Frtaj8', name='search', type='function_call', id='fc_00e27062616190e3006a60c1294ee081a38cb38d64e09d6a69', namespace=None, status='completed')]

In [15]:
print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course can I join"}', call_id='call_JDw6VPR211awUVlKGFPfpLbG', name='search', type='function_call', id='fc_075183cd4db223e0006a60c0ec380881a3b99bf48b8d1c796b', namespace=None, status='completed')]


Look at the output. Instead of a message with the answer, the response contains a function_call entry. 

This is a tool-call request, not the final answer.

The model decided to call your search function with:
{
    "query": "Can I join the course after it has started? discovered the course can I join"
}

Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like "enroll late join course".

The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

In [20]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join registration"}', call_id='call_X2FvALajrHjECxIPM4Frtaj8', name='search', type='function_call', id='fc_00e27062616190e3006a60c1294ee081a38cb38d64e09d6a69', namespace=None, status='completed')

In [19]:
response.output[0].arguments

'{"query":"Can I join the course if I just discovered it? enrollment late join registration"}'

In [32]:
response.output[0].call_id

'call_X2FvALajrHjECxIPM4Frtaj8'

status='completed' means the model finished generating the tool call—it does not mean your search function has run.

Execute the search and return its result to the model:

#### Executing the function and sending the result back.

The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

In [26]:
import json

call = response.output[0]
args = json.loads(call.arguments)

# Run your local search function
# 'search(**args)' will unpack the arguments from the call and pass them to the search function.
results = search(**args)    
result_json = json.dumps(results, indent=2)

In [23]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."\n  },\n  {\n    "id": "193612db63",\n    "course": "llm-zoomcamp",\n    "section": "Module 3: Orchestration",\n    "question": "Why do we need orchestration / Kestra 

Provide the search result to the model...

In [29]:
final_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=response.id,
    tools=[search_tool],
    input=[
        {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": str(result_json),
        }
    ],
)

print(final_response.output_text)

Yes — you can still join.

If you mean **following the course materials**, you can start anytime.  
If you want a **certificate**, you’ll need to submit your project while submissions are still open for the live cohort.


Alternative, equivalent, syntax:

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

In [ ]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,    # Use the call_id from the model's function call
    "output": result_json,
})

The model can see both what it requested and what the function returned, allowing it to formulate the final answer. extend() is used for response.output because it is a list; append() is used for the function result because it is one item.

In [31]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join registration"}', call_id='call_X2FvALajrHjECxIPM4Frtaj8', name='search', type='function_call', id='fc_00e27062616190e3006a60c1294ee081a38cb38d64e09d6a69', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_X2FvALajrHjECxIPM4Frtaj8',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I 

May now call the model again:

In [33]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join and start learning now.

One important caveat: if you want a certificate, you need to submit your project while the course is still accepting submissions.


This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper course-specific answer.

We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. That means the question, the decision to call search, and the result we got back.

That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. The LLM decides which tools to call.

### Token usage and cost
We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

The response has a usage field with the token counts:

In [34]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(980, 40)

In [38]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.75
    OUTPUT_PRICE_PER_MILLION = 4.50

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.000915
